In [83]:
import pandas as pd
import numpy as np

In [99]:
# 新增用户数据
user_df = pd.read_csv('D:\Desktop\神行少女数据\神行少女新增数据.csv',index_col=False)

In [100]:
# 用户闯关数据，包含通关、失败、主动离开
df1 = pd.read_csv('D:\Desktop\神行少女数据\神行少女闯关完成数据.csv',index_col=False)
df1 = df1[['设备ID','time','#dungeon_name','name']]
df2 = pd.read_csv('D:\Desktop\神行少女数据\神行少女闯关失败数据.csv',index_col=False)
df2 = df2[['设备ID','time','#dungeon_name','name']]
df3 = pd.read_csv('D:\Desktop\神行少女数据\神行少女闯关离开数据.csv',index_col=False)
df3 = df3[['设备ID','time','#dungeon_name','name']]
level_df = pd.concat((df1,df2,df3))

In [101]:
level_df.columns=['设备ID','游戏时间','关卡名称','事件名称']

In [102]:
user_df = user_df[['设备ID','time']]
user_df.columns=['设备ID','安装时间']

In [103]:
df = user_df.merge(level_df,how='left',on='设备ID')
df['游戏时间'] = pd.to_datetime(df['游戏时间'])
df['安装时间'] = pd.to_datetime(df['安装时间'])

In [104]:
df.shape

(115990, 5)

In [105]:
df['生命周期'] =  df.apply(lambda x:(x['游戏时间']- x['安装时间']).days if x['游戏时间']==x['游戏时间'] else 0,axis=1)

In [106]:
df.head()

,设备ID,安装时间,游戏时间,关卡名称,事件名称,生命周期
0,762a547ba9a799ee,2025-01-01 04:02:29.918,2025-01-01 04:28:53.666,1-2 未知力量,#dungeon_completed,0
1,762a547ba9a799ee,2025-01-01 04:02:29.918,2025-01-01 03:08:17.617,[普通]腐蚀迷森,#dungeon_fail,-1
2,4c1304abd99a2086,2025-01-01 04:13:56.398,2025-01-01 04:29:36.639,1-3 撤离许可,#dungeon_completed,0
3,762a547ba9a799ee,2025-01-01 04:14:43.880,2025-01-01 04:28:53.666,1-2 未知力量,#dungeon_completed,0
4,762a547ba9a799ee,2025-01-01 04:14:43.880,2025-01-01 03:08:17.617,[普通]腐蚀迷森,#dungeon_fail,-1


In [107]:
# 获取用户最后一次游戏的生命周期天数
df['rank_play']  = df.groupby(by=['设备ID'])['游戏时间'].rank(ascending=False,method='dense')
df = df.query('rank_play==1')

In [108]:
# 判断留存
df['1D'] = df.apply(lambda x: x['设备ID'] if x['生命周期'] >= 1 else np.nan,axis=1)
df['3D'] = df.apply(lambda x: x['设备ID'] if x['生命周期'] >= 3 else np.nan,axis=1)
df['7D'] = df.apply(lambda x: x['设备ID'] if x['生命周期'] >= 7 else np.nan,axis=1)

In [113]:
df['7D'].nunique()

475

In [120]:
# 次日留失用户流失节点 （主线）
df_3D=df.query('生命周期<7&生命周期>=3')
df_3D.to_excel('D:\Desktop\神行少女数据\神行少女7日流失数据.xlsx')

In [123]:
# 次日流失且有游戏行为用户的抽卡和武器补给数据

# 抽卡
gacha_df = pd.concat((pd.read_csv('D:\Desktop\神行少女数据\流失用户抽卡数据.csv',index_col=False),pd.read_csv('D:\Desktop\神行少女数据\流失用户抽卡数据2.csv',index_col=False)))


In [125]:
# 武器补给
weapon_df = pd.read_csv('D:\Desktop\神行少女数据\流失用户武器补给.csv',index_col=False)

In [126]:
weapon_df.head()

,name,time,设备ID
0,#weapon_gacha,2024-12-31 16:37:21.503,430acbaecc62e723
1,#weapon_gacha,2025-01-01 01:22:23.798,1E86B417-617A-4574-B42E-EBCE7A12A821
2,#weapon_gacha,2025-01-01 02:53:36.895,30dbe372c1bd3c1b
3,#weapon_gacha,2025-01-01 03:17:02.953,3a663f421690d583
4,#weapon_gacha,2025-01-01 03:30:04.037,eba4ec4060be7fd7


In [130]:
gacha_df1=df_1D[df_1D['关卡名称'].notnull()].merge(gacha_df,how='left',on='设备ID') 

In [132]:
gacha_df1['time'].nunique()

10780

In [133]:
weapon_df2=df_1D[df_1D['关卡名称'].notnull()].merge(weapon_df,how='left',on='设备ID') 

In [134]:
weapon_df2[weapon_df2['time'].notnull()]['设备ID'].nunique()

692

In [135]:
weapon_df2['time'].nunique()

1451